In [1]:
import numpy as np
import importlib
import scipy.sparse as sp
import utils
import torch
import random
import attack

importlib.reload(utils)
importlib.reload(attack)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
adj, features, labels = utils.load_data("cora")
print("adj information")
print(type(adj), adj.shape)
print(type(adj.data), adj.data.shape)
print("feature information")
print(type(features), features.shape)
print(features.dtype)
print(np.unique(features).shape)
print(features.min(), features.max())
print("labels information")
print(type(labels), labels.shape)
print(labels.dtype)
print(labels.min(), labels.max())

adj information
<class 'scipy.sparse._coo.coo_matrix'> (2708, 2708)
<class 'numpy.ndarray'> (5429,)
feature information
<class 'numpy.ndarray'> (2708, 1433)
float32
(2,)
0.0 1.0
labels information
<class 'numpy.ndarray'> (2708,)
int32
0 6


In [3]:
adj = adj + adj.T
adj[adj > 1] = 1
assert (adj != adj.T).nnz == 0 , "not undirected graph"
diag = adj.diagonal()
assert np.sum(diag!=0) == 0 , "have self_loop"

seed = 42
random.seed(seed)
np.random.seed(seed)

# 可以使用两种划分方式 parameter 1 or 2
idx_train, idx_val, idx_test = utils.split_data(labels, 1)
print(idx_train.shape, idx_val.shape, idx_test.shape)

(140,) (500,) (1000,)


In [4]:
adj_norm = utils.normalize_sparse_coo_adj(adj)
features_norm = utils.normalize_features(features)

adj_norm_tensor = utils.sparse_mx_to_torch_sparse_tensor(adj_norm)
features_norm_tensor = torch.FloatTensor(features_norm)
# print(adj_norm_tensor.dtype)
# print(features_norm_tensor.dtype)
labels_tensor = torch.LongTensor(labels)
idx_train_tensor = torch.LongTensor(idx_train)
idx_val_tensor = torch.LongTensor(idx_val)
idx_test_tensor = torch.LongTensor(idx_test)
print(type(adj_norm_tensor))

<class 'torch.Tensor'>


In [ ]:
module = attack.GCNSprase(adj_norm_tensor, features_norm_tensor, labels_tensor,
                          idx_train_tensor, idx_val_tensor, idx_test_tensor,
                          device=device,
                          dropout=0.5, lr=0.01, weight_decay=5e-4)

module.train(epochs=200)
pred = module.evalute()
print("acc=", np.sum(pred == labels) / len(labels))

array([6, 5, 4, ..., 1, 0, 2], dtype=int64)